# ML Error Analysis & Model Introspection Notebook

**Project:** Interview-ai-er  
**Artifact:** Production ML Error Diagnostic Suite  

Beyond single aggregate metrics (like Accuracy or macro F1), this notebook enables deep inspection of **categorized failure modes**, **confusion matrices**, **per-class tradeoffs**, **hard examples**, **false positives / false negatives**, and **confidence calibration distributions** across all five machine learning models:

1. **Question Difficulty Predictor** (`question-difficulty-v1`)
   * *Easy classified as medium*
   * *Medium classified as hard*
   * *Hard classified as easy*
2. **Question Skill Tagger** (`question-skill-v1`)
   * *Missed skill (FN)*
   * *Incorrect skill (FP)*
   * *Ambiguous / borderline tag*
3. **NLI Answer Concept Coverage Analyzer** (`answer-nli-v1`)
   * *False entailment (buzzword / tautology)*
   * *False contradiction (valid nuance / negation)*
   * *Partial coverage scoring errors*
4. **Code Defect & Risk Detector** (`code-risk-v1`)
   * *False vulnerability (defensive / clean code flagged)*
   * *Missed defect (real bug overlooked)*
5. **Candidate Mastery Model** (`mastery-v1`)
   * *Overestimation (lucky guess / small item inflation)*
   * *Underestimation (careless slip / cold-start penalty)*

In [ ]:
import json
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt

from ml.evaluation.error_analysis import (
    DifficultyErrorAnalyzer,
    SkillTaggerErrorAnalyzer,
    NLIAnswerAnalyzerErrorAnalyzer,
    CodeDefectErrorAnalyzer,
    MasteryErrorAnalyzer,
    run_all_model_error_analyses,
)

print("Environment initialized successfully. PyTorch, Sklearn, and ML components ready.")

## 1. Execute or Load Error Analysis Suite

We can run the live error analysis across all models or load previously generated reports from `reports/error_analysis/`.

In [ ]:
reports_dir = Path("reports/error_analysis")

if (reports_dir / "error_analysis_summary.json").exists():
    print("Loading cached error analysis reports from:", reports_dir.resolve())
    reports = {}
    # Load each report
    for model_key in ["difficulty", "skill_tagger", "nli_analyzer", "code_defect", "mastery"]:
        path = reports_dir / f"{model_key}_error_report.json"
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                reports[model_key] = json.load(f)
else:
    print("Running full error analysis suite now...")
    raw_reports = run_all_model_error_analyses(output_dir=str(reports_dir))
    reports = {k: v.to_dict() for k, v in raw_reports.items()}

print(f"Loaded {len(reports)} model reports: {list(reports.keys())}")

## 2. Visualization Utilities for Error Analysis

We construct plotting helpers for:
1. **Confusion Matrix Heatmaps**
2. **Per-Class Precision, Recall, and F1 Profiles**
3. **Confidence Distribution (Correct vs Errors)**
4. **Categorized Failure Mode Breakdown**

In [ ]:
def plot_confusion_matrix(cm_dict, title="Confusion Matrix"):
    labels = cm_dict["labels"]
    matrix = np.array(cm_dict["raw_matrix"])
    norm_matrix = np.array(cm_dict["normalized_matrix"])
    
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(norm_matrix, cmap="Blues", vmin=0, vmax=1)
    
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted Label", fontweight="bold")
    ax.set_ylabel("True Label", fontweight="bold")
    ax.set_title(title, fontweight="bold", pad=12)
    
    for i in range(len(labels)):
        for j in range(len(labels)):
            count = matrix[i, j]
            pct = norm_matrix[i, j] * 100
            text_color = "white" if norm_matrix[i, j] > 0.5 else "black"
            ax.text(j, i, f"{count}\n({pct:.0f}%)", ha="center", va="center", color=text_color, fontsize=10)
            
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Row Normalized Accuracy")
    plt.tight_layout()
    plt.show()

def plot_per_class_metrics(per_class_dict, title="Per-Class Performance"):
    classes = list(per_class_dict.keys())
    prec = [per_class_dict[c]["precision"] for c in classes]
    rec = [per_class_dict[c]["recall"] for c in classes]
    f1 = [per_class_dict[c]["f1_score"] for c in classes]
    
    x = np.arange(len(classes))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - width, prec, width, label="Precision", color="#3b82f6")
    ax.bar(x, rec, width, label="Recall", color="#10b981")
    ax.bar(x + width, f1, width, label="F1-Score", color="#8b5cf6")
    
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=30, ha="right")
    ax.set_ylabel("Score (0.0 - 1.0)", fontweight="bold")
    ax.set_ylim(0, 1.1)
    ax.set_title(title, fontweight="bold")
    ax.legend(loc="upper right")
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

def plot_confidence_distribution(cd_dict, title="Confidence Distribution"):
    edges = cd_dict["bin_edges"]
    correct = cd_dict["correct_counts"]
    error = cd_dict["error_counts"]
    
    x = np.arange(len(correct))
    bin_labels = [f"{edges[i]:.1f}-{edges[i+1]:.1f}" for i in range(len(edges)-1)]
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - 0.15, correct, width=0.3, label="Correct Predictions", color="#10b981")
    ax.bar(x + 0.15, error, width=0.3, label="Error Predictions", color="#ef4444")
    
    ax.set_xticks(x)
    ax.set_xticklabels(bin_labels, rotation=45, ha="right")
    ax.set_xlabel("Confidence Bin", fontweight="bold")
    ax.set_ylabel("Count", fontweight="bold")
    ax.set_title(f"{title}\nMean Correct: {cd_dict['mean_confidence_correct']:.2f} | Mean Error: {cd_dict['mean_confidence_error']:.2f} | Gap: {cd_dict['calibration_gap']:+.2f}", fontweight="bold")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

## 3. Question Difficulty Predictor (`question-difficulty-v1`)

### Failure Categories Inspected:
* **Easy classified as medium** (`easy_classified_as_medium`)
* **Medium classified as hard** (`medium_classified_as_hard`)
* **Hard classified as easy** (`hard_classified_as_easy`)

In [ ]:
diff_rep = reports["difficulty"]
print(f"Samples: {diff_rep['total_samples']} | Accuracy: {diff_rep['primary_metric']:.2%} | Macro F1: {diff_rep['macro_f1']:.3f}")

plot_confusion_matrix(diff_rep["confusion_matrix"], title="Question Difficulty Confusion Matrix")
plot_per_class_metrics(diff_rep["per_class_metrics"], title="Difficulty Predictor Per-Class Metrics")
plot_confidence_distribution(diff_rep["confidence_distribution"], title="Difficulty Confidence Distribution")

In [ ]:
print("=== CATEGORIZED DIFFICULTY FAILURE CASES ===")
for cat in ["easy_classified_as_medium", "medium_classified_as_hard", "hard_classified_as_easy"]:
    cases = [c for c in diff_rep["failure_cases"] if c["category"] == cat]
    print(f"\n-- [{cat.upper()}] ({len(cases)} cases) --")
    for c in cases[:3]:
        print(f"* ID: {c['id']}")
        print(f"  Question: {c['question']}")
        print(f"  Expected: {c['expected']} | Predicted: {c['predicted']} (Conf: {c['confidence']:.2f})")
        print(f"  Notes: {c['notes']}")

## 4. Question Skill Classifier (`question-skill-v1`)

### Failure Categories Inspected:
* **Missed skill** (`missed_skill` - False Negative)
* **Incorrect skill** (`incorrect_skill` - False Positive)
* **Ambiguous tag** (`ambiguous_tag` - Low margin between top predictions)

In [ ]:
skill_rep = reports["skill_tagger"]
print(f"Samples: {skill_rep['total_samples']} | Macro F1: {skill_rep['macro_f1']:.3f}")

plot_confusion_matrix(skill_rep["confusion_matrix"], title="Skill Classifier Aggregated Binary Confusion Matrix")
plot_per_class_metrics(skill_rep["per_class_metrics"], title="Per-Skill Precision, Recall & F1")
plot_confidence_distribution(skill_rep["confidence_distribution"], title="Skill Confidence Distribution")

In [ ]:
print("=== SKILL TAGGER CATEGORIZED FAILURE EXAMPLES ===")
for cat in ["missed_skill", "incorrect_skill", "ambiguous_tag"]:
    matching = [c for c in skill_rep["failure_cases"] if cat in c.get("categories", [])]
    print(f"\n-- Category: {cat.upper()} ({len(matching)} cases) --")
    for m in matching[:2]:
        print(f"* ID: {m['id']}")
        print(f"  Question: {m['question']}")
        print(f"  Expected: {m['expected_skills']}")
        print(f"  Predicted: {m['predicted_skills']} (Margin: {m.get('margin', 0.0):.2f})")
        print(f"  Context: {m.get('notes', '')}")

## 5. NLI Concept Coverage Analyzer (`answer-nli-v1`)

### Failure Categories Inspected:
* **False entailment** (`false_entailment` - Buzzwords marked as covered)
* **False contradiction** (`false_contradiction` - Valid nuance marked as contradiction)
* **Partial coverage errors** (`partial_coverage_errors` - Scoring calibration error)

In [ ]:
nli_rep = reports["nli_analyzer"]
print(f"Samples: {nli_rep['total_samples']} | Primary Metric: {nli_rep['primary_metric']:.2%}")

plot_confusion_matrix(nli_rep["confusion_matrix"], title="Concept Status Confusion Matrix")
plot_per_class_metrics(nli_rep["per_class_metrics"], title="Concept Coverage Status Metrics")
plot_confidence_distribution(nli_rep["confidence_distribution"], title="NLI Confidence Distribution")

In [ ]:
print("=== NLI ANSWER ANALYZER HARD EXAMPLES ===")
for ex in nli_rep["hard_examples"][:3]:
    print(f"* ID: {ex['example_id']} | Category: {ex['error_category']} | Conf: {ex['confidence']:.2f}")
    print(f"  Snippet: {ex['input_snippet'][:120]}...")
    print(f"  Explanation: {ex['root_cause_explanation']}\n")

## 6. Code Defect & Risk Detector (`code-risk-v1`)

### Failure Categories Inspected:
* **False vulnerability** (`false_vulnerability` - Clean code falsely flagged, FP)
* **Missed defect** (`missed_defect` - Real bug overlooked, FN)

In [ ]:
code_rep = reports["code_defect"]
print(f"Samples: {code_rep['total_samples']} | Accuracy: {code_rep['primary_metric']:.2%} | Macro F1: {code_rep['macro_f1']:.3f}")

plot_confusion_matrix(code_rep["confusion_matrix"], title="Code Defect Detector Confusion Matrix")
plot_per_class_metrics(code_rep["per_class_metrics"], title="Defect Detector Per-Class Metrics")
plot_confidence_distribution(code_rep["confidence_distribution"], title="Code Risk Confidence Distribution")

In [ ]:
print("=== CODE DEFECT FALSE POSITIVES (FALSE VULNERABILITIES) ===")
for fp in code_rep["false_positives"][:3]:
    print(f"* ID: {fp['id']} | Defect Prob: {fp['defect_prob']:.2f}")
    print(f"  Code:\n{fp['code']}")
    print(f"  Notes: {fp['notes']}\n")

print("=== CODE DEFECT FALSE NEGATIVES (MISSED DEFECTS) ===")
for fn in code_rep["false_negatives"][:3]:
    print(f"* ID: {fn['id']} | Defect Prob: {fn['defect_prob']:.2f}")
    print(f"  Code:\n{fn['code']}")
    print(f"  Notes: {fn['notes']}\n")

## 7. Candidate Mastery Model (`mastery-v1`)

### Failure Categories Inspected:
* **Overestimation** (`overestimation` - Lucky guess / small sample inflation)
* **Underestimation** (`underestimation` - Careless slip / cold-start drag)

In [ ]:
mastery_rep = reports["mastery"]
print(f"Candidate Trajectories: {mastery_rep['total_samples']} | Tier Accuracy: {mastery_rep['primary_metric']:.2%}")

plot_confusion_matrix(mastery_rep["confusion_matrix"], title="Mastery Model Ability Tier Confusion Matrix")
plot_per_class_metrics(mastery_rep["per_class_metrics"], title="Mastery Tier Classification Metrics")
plot_confidence_distribution(mastery_rep["confidence_distribution"], title="Mastery Confidence / Uncertainty Distribution")

In [ ]:
print("=== MASTERY OVERESTIMATION & UNDERESTIMATION CASES ===")
for fc in mastery_rep["failure_cases"]:
    print(f"* Profile: {fc['profile']} ({fc['category'].upper()})")
    print(f"  True θ: {fc['true_theta']:+.2f} ({fc['true_tier']}) | Estimated θ: {fc['estimated_theta']:+.2f} ({fc['estimated_tier']})")
    print(f"  Error: {fc['theta_error']:+.2f} | Notes: {fc['notes']}")
    print("  Interaction Trajectory:")
    for step in fc['interactions']:
        print(f"    Diff: {step['difficulty']:<12} Score: {step['score']:<5} -> Theta: {step['resulting_theta']:+.2f} (Proficiency: {step['proficiency']}%)")
    print()